
# RetailPulse Silver Layer

Purpose:
Transform Bronze data into trusted business-ready data.

Operations:
- Join Customers
- Join Products
- Calculate Revenue

Output:
- silver_sales_enriched


## Read Bronze Tables

In [0]:
bronze_customers_df = spark.table(
    "workspace.retailpulse.bronze_customers"
)
bronze_products_df = spark.table(
    "workspace.retailpulse.bronze_products"
)
bronze_sales_df = spark.table(
    "workspace.retailpulse.bronze_sales"
)
bronze_returns_df = spark.table(
    "workspace.retailpulse.bronze_returns"
)
display(bronze_customers_df)
display(bronze_products_df)
display(bronze_sales_df)
display(bronze_returns_df)


### Join Sales + Customers

In [0]:
sales_customers_df = (
    bronze_sales_df.join(
        bronze_customers_df,
        on = "customer_id",
        how = "inner"
    )
)

display(sales_customers_df)


### Join Products

In [0]:
silver_sales_enriched_df = (sales_customers_df.
                            join(
                                bronze_products_df,
                                on = "product_id",
                                how = "inner"
                            )
                            )
display(silver_sales_enriched_df)

### Create Revenue Column

In [0]:
from pyspark.sql.functions import col

silver_sales_enriched_df = (
    silver_sales_enriched_df.withColumn(
        "revenue", col("quantity") * col("price")
    )
)

display(silver_sales_enriched_df)


### Create Silver Table

- Now Let's Make It a Real Silver Table
- silver_sales_enriched_df - exists only in memory.

If you detach and reattach the notebook:
- POOF 💨
- gone

We need to persist it as a Delta table.

In [0]:
silver_sales_enriched_df.write.format("delta") \
    .mode("overwrite").saveAsTable(
        "workspace.retailpulse.silver_sales_enriched"
    )

In [0]:
%sql

SHOW TABLES IN workspace.retailpulse;

SILVER
──────────────────

silver_sales_enriched

(Joined + Revenue)

In [0]:
%sql

SELECT * FROM workspace.retailpulse.silver_sales_enriched;

In [0]:
%sql

DESCRIBE DETAIL workspace.retailpulse.silver_sales_enriched

In [0]:
# Now create the DF using the Already created TABLE:

silver_df = spark.table("workspace.retailpulse.silver_sales_enriched")

In [0]:
display(silver_df)
# DBTITLE 1

In [0]:
%sql
SELECT *
FROM workspace.retailpulse.silver_sales_enriched